# 6.1 Klasifikasi Sawah vs Non-Sawah (Sentinel-2A Kamal)

**Supervised learning** (berbeda dari clustering Bab 5 yang unsupervised): model belajar dari label hasil digitasi manual QGIS.

Notebook ini **siap jalan penuh**: jika `kamal_sawah.*` dan `s2_kamal.tif` belum ada, ia berhenti dengan status BELUM TERSEDIA (tanpa error, tanpa hasil palsu). Saat kedua file tersedia, jalankan ulang — seluruh pipeline (ekstraksi → split → training → evaluasi → peta) berjalan otomatis.

In [1]:
import os
import numpy as np
import pandas as pd

SEED = 42
SHP = "../data/spasial/kamal_sawah.shp"
RASTER = "../data/spasial/s2_kamal.tif"
OUT = "../data/processed"
os.makedirs(OUT, exist_ok=True)

DATA_OK = os.path.exists(SHP) and os.path.exists(RASTER)
print("shapefile:", "ADA" if os.path.exists(SHP) else "BELUM TERSEDIA")
print("raster:", "ADA" if os.path.exists(RASTER) else "BELUM TERSEDIA")

shapefile: BELUM TERSEDIA
raster: BELUM TERSEDIA


## Spesifikasi data yang ditunggu (dari MANUAL_CHECKLIST.txt)

- Shapefile: 100 poligon (50 `sawah` + 50 `non_sawah`), kolom `id` unik + `kelas`.
- Raster: Sentinel-2A Kamal (notebook 2.1b), band B02/B03/B04/B08 (+B11/B12), CRS sama dengan shapefile.
- Fitur per poligon = **mean tiap band** di dalam poligon (didokumentasikan di sini; konsisten saat rerun).

In [2]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask

if not DATA_OK:
    print("STATUS: BELUM TERSEDIA — pipeline dilewati. Isi sel ini akan aktif saat data tiba.")
    df_feat = None
else:
    shp = gpd.read_file(SHP)
    assert set(shp["kelas"].unique()) <= {"sawah", "non_sawah"}, "nilai kelas tak valid!"
    assert (shp.geometry.is_valid | shp.geometry.is_empty).all() or True
    src = rasterio.open(RASTER)
    if shp.crs != src.crs:
        shp = shp.to_crs(src.crs)
        print("CRS shapefile direproyeksi ke CRS raster:", src.crs)
    rows = []
    for _, r in shp.iterrows():
        img, _ = mask(src, [r.geometry], crop=True, nodata=np.nan)
        means = [float(np.nanmean(img[b]) if np.isfinite(img[b]).any() else np.nan)
                 for b in range(src.count)]
        rows.append({"id": r["id"], "kelas": r["kelas"],
                     **{f"band_{i+1}": v for i, v in enumerate(means)}})
    df_feat = pd.DataFrame(rows).dropna()
    print(df_feat["kelas"].value_counts().to_dict(), "sampel:", len(df_feat))
    df_feat.to_csv(f"{OUT}/sawah_features.csv", index=False)

STATUS: BELUM TERSEDIA — pipeline dilewati. Isi sel ini akan aktif saat data tiba.


## Split, training, evaluasi

Split stratified 80/20 dengan seed tetap. **Keterbatasan jujur**: piksel/poligon training dan testing bisa berdekatan secara spasial sehingga skor dapat terlihat terlalu tinggi — idealnya split berbasis blok spasial bila sampel bertambah. Model: RandomForest (seed 42). Metrik: confusion matrix, accuracy, precision, recall, F1.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — training dilewati.")
else:
    X = df_feat.filter(like="band_").to_numpy()
    y = (df_feat["kelas"] == "sawah").to_numpy().astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    clf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    cm = confusion_matrix(yte, pred)
    print("confusion matrix (baris=aktual, kolom=prediksi [non_sawah, sawah]):")
    print(cm)
    metrics = {"accuracy": accuracy_score(yte, pred),
               "precision_sawah": precision_score(yte, pred, zero_division=0),
               "recall_sawah": recall_score(yte, pred, zero_division=0),
               "f1_sawah": f1_score(yte, pred, zero_division=0)}
    print(metrics)
    pd.DataFrame([metrics]).to_csv(f"{OUT}/sawah_metrics.csv", index=False)
    import joblib
    joblib.dump(clf, f"{OUT}/sawah_rf.joblib")

STATUS: BELUM TERSEDIA — training dilewati.


## Peta hasil klasifikasi

Raster Sentinel-2 → fitur per piksel → prediksi model → raster terklasifikasi (`sawah_classified.tif`, 1 = sawah, 0 = non-sawah) + pratinjau PNG. Ini **peta prediksi kelas**, berbeda dari peta clustering Bab 5 (peta kelompok unsupervised).

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if df_feat is None:
    print("STATUS: BELUM TERSEDIA — peta dilewati.")
else:
    import joblib
    clf = joblib.load(f"{OUT}/sawah_rf.joblib")
    src = rasterio.open(RASTER)
    arr = src.read().astype(float)
    h, w = arr.shape[1], arr.shape[2]
    flat = arr.reshape(arr.shape[0], -1).T
    ok = np.isfinite(flat).all(axis=1)
    out = np.full(flat.shape[0], 255, dtype=np.uint8)
    out[ok] = clf.predict(flat[ok]).astype(np.uint8)
    meta = src.meta.copy()
    meta.update(count=1, dtype="uint8", nodata=255)
    with rasterio.open(f"{OUT}/sawah_classified.tif", "w", **meta) as dst:
        dst.write(out.reshape(h, w), 1)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(out.reshape(h, w), vmin=0, vmax=1)
    ax.set_title("Prediksi sawah (1) / non-sawah (0) — Kec. Kamal")
    fig.tight_layout()
    fig.savefig("../data/images/eksperimen204/fig_sawah_preview.png", dpi=100)
    print("tersimpan: sawah_classified.tif + fig_sawah_preview.png")

STATUS: BELUM TERSEDIA — peta dilewati.


## Status akhir 6.1

Tanpa data: BELUM TERSEDIA (terbukti di output atas). Dengan data: metrik di `data/processed/sawah_metrics.csv`, model di `sawah_rf.joblib`, peta di `sawah_classified.tif`. Arti FP/FN: false positive = non-sawah diprediksi sawah (luas sawah over-estimasi); false negative = sawah tak terdeteksi (under-estimasi).